# Supply Chain Logistics Problem — Dataset Analysis

**Source:** [Kaggle — Supply Chain Logistics Problem](https://www.kaggle.com/datasets/anisseezzebdi/supply-chain-logistics-problem) (originally published by researchers at Brunel University, based on demand data from a global microchip producer)

## What this dataset represents

A microchip manufacturer operates **19 warehouses**, ships through **11 origin ports** to **1 destination port**, and serves many customers. For every customer order, the company must decide:

> *Which warehouse fulfils it, through which port, using which carrier and service level — at what cost?*

This decision is constrained: not every warehouse stocks every product, not every warehouse may serve every customer, not every warehouse connects to every port, and not every carrier serves every port pair. The dataset captures this network as **7 related tables** rather than one flat file — historical order demand plus the cost/capacity/connectivity rules that govern feasible fulfilment.

| Category | Question it answers | Table(s) |
|---|---|---|
| Demand | What was ordered? | `OrderList` |
| Transportation cost | How much does shipping cost? | `FreightRates` |
| Warehouse cost | How much does handling cost? | `WhCosts` |
| Warehouse capacity | How much can a warehouse process? | `WhCapacities` |
| Product restriction | Which warehouse stocks which product? | `ProductsPerPlant` |
| Customer restriction | Which warehouse may serve which customer? | `VmiCustomers` |
| Network connectivity | Which ports can a warehouse ship from? | `PlantPorts` |

This notebook loads each table, reports its **shape (rows × columns)**, inspects its structure (dtypes, nulls, sample rows), and explains what it represents in the context of the overall supply-chain network optimization problem.


## 0. Setup

Load every sheet of the Excel workbook into its own `pandas.DataFrame`.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_PATH = "../data/Supply chain logistics problem.xlsx"

xls = pd.ExcelFile(DATA_PATH)
print("Sheets found:", xls.sheet_names)

Sheets found: ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']


In [2]:
# Load every table once
order_list        = pd.read_excel(xls, "OrderList")
freight_rates      = pd.read_excel(xls, "FreightRates")
wh_costs           = pd.read_excel(xls, "WhCosts")
wh_capacities       = pd.read_excel(xls, "WhCapacities")
products_per_plant  = pd.read_excel(xls, "ProductsPerPlant")
vmi_customers       = pd.read_excel(xls, "VmiCustomers")
plant_ports         = pd.read_excel(xls, "PlantPorts")

tables = {
    "OrderList": order_list,
    "FreightRates": freight_rates,
    "WhCosts": wh_costs,
    "WhCapacities": wh_capacities,
    "ProductsPerPlant": products_per_plant,
    "VmiCustomers": vmi_customers,
    "PlantPorts": plant_ports,
}

In [3]:
# Quick overview: rows and columns for every table
overview = pd.DataFrame(
    [(name, df.shape[0], df.shape[1], list(df.columns)) for name, df in tables.items()],
    columns=["Table", "Rows", "Columns", "Column names"],
)
overview

,Table,Rows,Columns,Column names
0,OrderList,9215,14,"[Order ID, Order Date, Origin Port, Carrier, T..."
1,FreightRates,1540,11,"[Carrier, orig_port_cd, dest_port_cd, minm_wgh..."
2,WhCosts,19,2,"[WH, Cost/unit]"
3,WhCapacities,19,2,"[Plant ID, Daily Capacity ]"
4,ProductsPerPlant,2036,2,"[Plant Code, Product ID]"
5,VmiCustomers,14,2,"[Plant Code, Customers]"
6,PlantPorts,22,2,"[Plant Code, Port]"


## 1. `OrderList` — Historical Demand

**What it represents:** The core demand table. Each of its rows is **one historical customer order**: what was ordered, by whom, how much it weighed, and how it was historically routed and fulfilled (which warehouse, port, carrier and service level were used).

This is *historical* routing data — it shows what the company actually did, not necessarily the cost-optimal choice. The other six tables describe the rules and costs of the network, which let us evaluate whether a *better* routing exists for this same demand.

| Column | Meaning |
|---|---|
| `Order ID` | Unique identifier for the order |
| `Order Date` | Date the order was placed |
| `Origin Port` | Port the shipment was routed through |
| `Carrier` | Carrier/courier historically used |
| `TPT` | Transportation time (transit days) |
| `Service Level` | `DTD` (Door-to-Door), `DTP` (Door-to-Port), or `CRF` (Customer Referred Freight — customer arranges their own transport, so no transportation cost applies) |
| `Ship ahead day count` | Days the shipment could go out ahead of schedule |
| `Ship Late Day count` | Days the shipment was allowed to be late |
| `Customer` | Customer who placed the order |
| `Product ID` | Product ordered |
| `Plant Code` | Warehouse that historically fulfilled the order |
| `Destination Port` | Final destination port |
| `Unit quantity` | Number of units ordered (drives warehouse handling cost) |
| `Weight` | Shipment weight (drives freight cost via weight bands) |


In [4]:
print(f"Shape: {order_list.shape[0]:,} rows x {order_list.shape[1]} columns")
order_list.info()

Shape: 9,215 rows x 14 columns
<class 'pandas.DataFrame'>
RangeIndex: 9215 entries, 0 to 9214
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Order ID              9215 non-null   float64       
 1   Order Date            9215 non-null   datetime64[us]
 2   Origin Port           9215 non-null   str           
 3   Carrier               9215 non-null   str           
 4   TPT                   9215 non-null   int64         
 5   Service Level         9215 non-null   str           
 6   Ship ahead day count  9215 non-null   int64         
 7   Ship Late Day count   9215 non-null   int64         
 8   Customer              9215 non-null   str           
 9   Product ID            9215 non-null   int64         
 10  Plant Code            9215 non-null   str           
 11  Destination Port      9215 non-null   str           
 12  Unit quantity         9215 non-null   int64         
 13

In [5]:
order_list.head()

,Order ID,Order Date,Origin Port,Carrier,TPT,Service Level,Ship ahead day count,Ship Late Day count,Customer,Product ID,Plant Code,Destination Port,Unit quantity,Weight
0,1.447296e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,808,14.30
1,1.447158e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,3188,87.94
2,1.447139e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2331,61.20
3,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,847,16.16
4,1.447364e+09,2013-05-26,PORT09,V44_3,1,CRF,3,0,V55555_53,1700106,PLANT16,PORT09,2163,52.34


In [6]:
print("Missing values per column:")
print(order_list.isna().sum())
print()
print("Unique customers:", order_list['Customer'].nunique())
print("Unique products:", order_list['Product ID'].nunique())
print("Unique plants (warehouses) used:", order_list['Plant Code'].nunique())
print("Unique origin ports:", order_list['Origin Port'].nunique())
print("Unique destination ports:", order_list['Destination Port'].nunique())
print("Unique carriers:", order_list['Carrier'].nunique())
print()
print("Service level breakdown:")
print(order_list['Service Level'].value_counts())

Missing values per column:
Order ID                0
Order Date              0
Origin Port             0
Carrier                 0
TPT                     0
Service Level           0
Ship ahead day count    0
Ship Late Day count     0
Customer                0
Product ID              0
Plant Code              0
Destination Port        0
Unit quantity           0
Weight                  0
dtype: int64

Unique customers: 46
Unique products: 772
Unique plants (warehouses) used: 7
Unique origin ports: 3
Unique destination ports: 1
Unique carriers: 3

Service level breakdown:
Service Level
DTP    6218
DTD    2143
CRF     854
Name: count, dtype: int64


## 2. `FreightRates` — Transportation Cost Rules

**What it represents:** Every available shipping option between a port pair, for a given carrier, service level and weight band, and what it costs. This is the pricing table used to compute transportation cost once a warehouse/port/carrier combination is chosen for an order.

Rates are **weight-banded** — the same carrier can charge a different `rate` per kg depending on which `[minm_wgh_qty, max_wgh_qty]` bracket the shipment weight falls in — and every quote has a `minimum cost` floor:

$$\text{Transportation Cost} = \max(\text{rate} \times \text{weight},\ \text{minimum cost})$$

| Column | Meaning |
|---|---|
| `Carrier` | Carrier/courier code |
| `orig_port_cd` | Origin port |
| `dest_port_cd` | Destination port |
| `minm_wgh_qty` / `max_wgh_qty` | Weight band this rate applies to |
| `svc_cd` | Service level code |
| `minimum cost` | Minimum charge for the shipment, regardless of weight |
| `rate` | Cost per unit weight |
| `mode_dsc` | Transportation mode (`AIR` / `GROUND`) |
| `tpt_day_cnt` | Estimated transit days |
| `Carrier type` | Carrier category/classification |


In [7]:
print(f"Shape: {freight_rates.shape[0]:,} rows x {freight_rates.shape[1]} columns")
freight_rates.info()

Shape: 1,540 rows x 11 columns
<class 'pandas.DataFrame'>
RangeIndex: 1540 entries, 0 to 1539
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Carrier       1540 non-null   str    
 1   orig_port_cd  1540 non-null   str    
 2   dest_port_cd  1540 non-null   str    
 3   minm_wgh_qty  1540 non-null   float64
 4   max_wgh_qty   1540 non-null   float64
 5   svc_cd        1540 non-null   str    
 6   minimum cost  1540 non-null   float64
 7   rate          1540 non-null   float64
 8   mode_dsc      1540 non-null   str    
 9   tpt_day_cnt   1540 non-null   int64  
 10  Carrier type  1540 non-null   str    
dtypes: float64(4), int64(1), str(6)
memory usage: 132.5 KB


In [8]:
freight_rates.head()

,Carrier,orig_port_cd,dest_port_cd,minm_wgh_qty,max_wgh_qty,svc_cd,minimum cost,rate,mode_dsc,tpt_day_cnt,Carrier type
0,V444_6,PORT08,PORT09,250.0,499.99,DTD,43.2272,0.7132,AIR,2,V88888888_0
1,V444_6,PORT08,PORT09,65.0,69.99,DTD,43.2272,0.7512,AIR,2,V88888888_0
2,V444_6,PORT08,PORT09,60.0,64.99,DTD,43.2272,0.7892,AIR,2,V88888888_0
3,V444_6,PORT08,PORT09,50.0,54.99,DTD,43.2272,0.8272,AIR,2,V88888888_0
4,V444_6,PORT08,PORT09,35.0,39.99,DTD,43.2272,1.0552,AIR,2,V88888888_0


In [9]:
print("Missing values per column:")
print(freight_rates.isna().sum())
print()
print("Unique carriers:", freight_rates['Carrier'].nunique())
print("Unique origin ports:", freight_rates['orig_port_cd'].nunique())
print("Unique destination ports:", freight_rates['dest_port_cd'].nunique())
print()
print("Transportation mode breakdown:")
print(freight_rates['mode_dsc'].value_counts())
print()
print("Service level breakdown:")
print(freight_rates['svc_cd'].value_counts())

Missing values per column:
Carrier         0
orig_port_cd    0
dest_port_cd    0
minm_wgh_qty    0
max_wgh_qty     0
svc_cd          0
minimum cost    0
rate            0
mode_dsc        0
tpt_day_cnt     0
Carrier type    0
dtype: int64

Unique carriers: 9
Unique origin ports: 10
Unique destination ports: 1

Transportation mode breakdown:
mode_dsc
AIR       1511
GROUND      29
Name: count, dtype: int64

Service level breakdown:
svc_cd
DTD    1322
DTP     218
Name: count, dtype: int64


## 3. `WhCosts` — Warehouse Handling Cost

**What it represents:** The per-unit handling/storage cost for each of the 19 warehouses. Combined with an order's `Unit quantity`, this gives the warehouse-side cost of fulfilling that order:

$$\text{Warehouse Cost} = \text{Unit quantity} \times \text{Cost/unit}$$

| Column | Meaning |
|---|---|
| `WH` | Warehouse identifier (matches `Plant Code` elsewhere) |
| `Cost/unit` | Cost of handling one unit of product at this warehouse |


In [10]:
print(f"Shape: {wh_costs.shape[0]:,} rows x {wh_costs.shape[1]} columns")
wh_costs.info()

Shape: 19 rows x 2 columns
<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   WH         19 non-null     str    
 1   Cost/unit  19 non-null     float64
dtypes: float64(1), str(1)
memory usage: 436.0 bytes


In [11]:
wh_costs.sort_values('Cost/unit')

,WH,Cost/unit
8,PLANT07,0.371424
18,PLANT04,0.428503
1,PLANT17,0.428947
13,PLANT09,0.465071
15,PLANT13,0.469707
4,PLANT02,0.477504
3,PLANT05,0.488144
7,PLANT10,0.493582
14,PLANT03,0.517502
17,PLANT08,0.522857


## 4. `WhCapacities` — Warehouse Throughput Limit

**What it represents:** The maximum number of **orders** each warehouse can process per day. This is a hard capacity constraint — note it limits the *count of orders*, not the number of product units, so a warehouse handling a few huge orders is treated the same as one handling the same number of small orders.

| Column | Meaning |
|---|---|
| `Plant ID` | Warehouse identifier (matches `Plant Code` elsewhere) |
| `Daily Capacity` | Maximum orders this warehouse can process per day |


In [12]:
print(f"Shape: {wh_capacities.shape[0]:,} rows x {wh_capacities.shape[1]} columns")
wh_capacities.info()

Shape: 19 rows x 2 columns
<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Plant ID         19 non-null     str  
 1   Daily Capacity   19 non-null     int64
dtypes: int64(1), str(1)
memory usage: 436.0 bytes


In [13]:
wh_capacities.sort_values('Daily Capacity ', ascending=False)

,Plant ID,Daily Capacity
5,PLANT01,1070
14,PLANT03,1013
18,PLANT04,554
9,PLANT14,549
15,PLANT13,490
10,PLANT16,457
3,PLANT05,385
12,PLANT11,332
8,PLANT07,265
11,PLANT12,209


## 5. `ProductsPerPlant` — Product-to-Warehouse Feasibility

**What it represents:** A compatibility matrix stating which products each warehouse is capable of supplying. This is a feasibility filter: an order for a given `Product ID` can only be routed through a warehouse (`Plant Code`) that appears in this table for that product — it cannot be assigned to a warehouse that doesn't stock it.

| Column | Meaning |
|---|---|
| `Plant Code` | Warehouse identifier |
| `Product ID` | Product this warehouse can supply |


In [14]:
print(f"Shape: {products_per_plant.shape[0]:,} rows x {products_per_plant.shape[1]} columns")
products_per_plant.info()

Shape: 2,036 rows x 2 columns
<class 'pandas.DataFrame'>
RangeIndex: 2036 entries, 0 to 2035
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Plant Code  2036 non-null   str  
 1   Product ID  2036 non-null   int64
dtypes: int64(1), str(1)
memory usage: 31.9 KB


In [15]:
print("Unique plants:", products_per_plant['Plant Code'].nunique())
print("Unique products:", products_per_plant['Product ID'].nunique())
print()
print("Number of products each plant can supply:")
products_per_plant.groupby('Plant Code')['Product ID'].nunique().sort_values(ascending=False)

Unique plants: 19
Unique products: 1540

Number of products each plant can supply:


Plant Code
PLANT03    781
PLANT01    220
PLANT13    150
PLANT04    134
PLANT05    127
PLANT10    121
PLANT02    116
PLANT16    113
PLANT11     96
PLANT12     57
PLANT07     29
PLANT06     26
PLANT08     21
PLANT17     20
PLANT18     12
PLANT09      8
PLANT14      3
PLANT15      1
CND9         1
Name: Product ID, dtype: int64

## 6. `VmiCustomers` — Vendor-Managed-Inventory Customer Restrictions

**What it represents:** Special warehouse-to-customer relationships (Vendor Managed Inventory). Where a warehouse/customer pair is listed here, that link reflects a dedicated servicing restriction between the two; warehouses not tied to a listed restriction remain free to serve customers generally. This is a small table (14 rows) covering only the customers subject to this special arrangement.

| Column | Meaning |
|---|---|
| `Plant Code` | Warehouse identifier |
| `Customers` | Customer this warehouse has a VMI relationship with |


In [16]:
print(f"Shape: {vmi_customers.shape[0]:,} rows x {vmi_customers.shape[1]} columns")
vmi_customers.info()

Shape: 14 rows x 2 columns
<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Plant Code  14 non-null     str  
 1   Customers   14 non-null     str  
dtypes: str(2)
memory usage: 356.0 bytes


In [17]:
vmi_customers

,Plant Code,Customers
0,PLANT02,V5555555555555_16
1,PLANT02,V555555555555555_29
2,PLANT02,V555555555_3
3,PLANT02,V55555555555555_8
4,PLANT02,V55555555_9
5,PLANT02,V55555_10
6,PLANT02,V55555555_5
7,PLANT06,V555555555555555_18
8,PLANT06,V55555_10
9,PLANT10,V555555555555555_29


## 7. `PlantPorts` — Warehouse-to-Port Connectivity

**What it represents:** The physical network topology: which origin ports each warehouse is actually connected to for shipping out product. An order can only leave a warehouse through a port listed for that warehouse here.

| Column | Meaning |
|---|---|
| `Plant Code` | Warehouse identifier |
| `Port` | Origin port this warehouse can ship from |


In [18]:
print(f"Shape: {plant_ports.shape[0]:,} rows x {plant_ports.shape[1]} columns")
plant_ports.info()

Shape: 22 rows x 2 columns
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Plant Code  22 non-null     str  
 1   Port        22 non-null     str  
dtypes: str(2)
memory usage: 484.0 bytes


In [19]:
print("Ports available per plant:")
plant_ports.groupby('Plant Code')['Port'].apply(list)

Ports available per plant:


Plant Code
PLANT01    [PORT01, PORT02]
PLANT02            [PORT03]
PLANT03            [PORT04]
PLANT04            [PORT05]
PLANT05            [PORT06]
PLANT06            [PORT06]
PLANT07    [PORT01, PORT02]
PLANT08            [PORT04]
PLANT09            [PORT04]
PLANT10    [PORT01, PORT02]
PLANT11            [PORT04]
PLANT12            [PORT04]
PLANT13            [PORT04]
PLANT14            [PORT07]
PLANT15            [PORT08]
PLANT16            [PORT09]
PLANT17            [PORT10]
PLANT18            [PORT11]
PLANT19            [PORT04]
Name: Port, dtype: object

## 8. Summary — How the Tables Fit Together

| Table | Rows | Columns | Role |
|---|---:|---:|---|
| `OrderList` | 9,215 | 14 | Demand — what needs to be delivered |
| `FreightRates` | 1,540 | 11 | Cost — how transportation is priced |
| `WhCosts` | 19 | 2 | Cost — per-unit warehouse handling cost |
| `WhCapacities` | 19 | 2 | Constraint — max orders/day per warehouse |
| `ProductsPerPlant` | 2,036 | 2 | Constraint — product/warehouse feasibility |
| `VmiCustomers` | 14 | 2 | Constraint — customer/warehouse restriction |
| `PlantPorts` | 22 | 2 | Constraint — warehouse/port connectivity |

**Reading order for a single order:**

1. `OrderList` gives the order's `Product ID`, `Customer`, `Unit quantity`, `Weight`.
2. `ProductsPerPlant` narrows candidate warehouses to those stocking that product.
3. `VmiCustomers` further restricts candidates to warehouses allowed to serve that customer.
4. `PlantPorts` gives the origin ports each candidate warehouse can ship from.
5. `FreightRates` gives the carriers, weight-banded rates and minimum charges available for each origin→destination port pair.
6. `WhCosts` (× `Unit quantity`) and the chosen `FreightRates` row together give total cost for that warehouse/port/carrier combination.
7. `WhCapacities` caps how many orders a warehouse may take in total.

The optimization task this dataset supports is: for every order in `OrderList`, choose a **warehouse → origin port → carrier → service level** combination that is *feasible* under constraints 2–4 and 7, minimizing:

$$\text{Total Cost} = \sum \text{Warehouse Cost} + \sum \text{Transportation Cost}$$

This is the real-world microchip-producer network the original researchers modeled with Ant Colony Optimization; it is equally suitable for a Mixed-Integer Linear Program or other metaheuristics.
